# RoboOps — DETR Round 1 Training

**Model:** `microsoft/conditional-detr-resnet-50`  
**Stage:** 1 — Coarse (backbone frozen, 480px, ~50 epochs)  
**Data:** Streams MDS shards from S3 via `mds_path` in `params.yaml`  
**Tracking:** DagsHub MLflow  

### Before running:
1. Runtime → Change runtime type → **A100 GPU**
2. Add Colab Secrets (🔑 left panel):
   - `AWS_ACCESS_KEY_ID`
   - `AWS_SECRET_ACCESS_KEY`
   - `DAGSHUB_TOKEN`
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch to GPU runtime!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU not available! Runtime → Change runtime type → A100'

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────
%%capture
!pip install transformers==4.40.0 torch torchvision \
             mlflow==2.12.1 dagshub==0.3.25 \
             optimum[onnxruntime]==1.19.0 \
             mosaicml-streaming==0.7.5 \
             pyyaml pycocotools boto3 accelerate Pillow
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Auth + clone repo ─────────────────────────────────────────────
import os
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = 'eu-central-1'

import dagshub, mlflow
dagshub.init(
    repo_owner='srnortw',
    repo_name='learning-robotic-perception-robops',
    mlflow=True
)
mlflow.set_experiment('detr-round1')
print('Auth complete. MLflow → DagsHub')

In [ ]:
# ── Cell 4: Clone repo + load params ─────────────────────────────────────
!git clone https://github.com/srnortw/learning-robotic-perception-robops.git
%cd learning-robotic-perception-robops

import yaml
with open('pipeline/phase_c/detr/params.yaml') as f:
    params = yaml.safe_load(f)

mds_path        = params['dataset']['mds_path']
dataset_version = params['dataset']['dataset_version']
num_classes     = params['dataset']['num_classes']
image_size      = params['preprocessing']['image_size']
mean            = params['preprocessing']['normalize_mean']
std             = params['preprocessing']['normalize_std']
epochs          = params['training']['epochs']
lr              = params['training']['learning_rate']

print(f'Dataset:  {mds_path}')
print(f'Version:  {dataset_version}')
print(f'Classes:  {num_classes}')
print(f'Epochs:   {epochs}')

In [ ]:
# ── Cell 5: Build DataLoaders (streams MDS shards from S3) ───────────────
from streaming import StreamingDataset
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image as PILImage
import io, torch

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

class DETRStreamingDataset(StreamingDataset):
    def __getitem__(self, idx):
        sample = super().__getitem__(idx)
        img = PILImage.open(io.BytesIO(sample['image'])).convert('RGB')
        return transform(img), sample['annotations']

def collate_fn(batch):
    images, annotations = zip(*batch)
    return torch.stack(images), list(annotations)

train_ds = DETRStreamingDataset(local='/tmp/mds/train', remote=mds_path + 'train/', shuffle=True)
val_ds   = DETRStreamingDataset(local='/tmp/mds/val',   remote=mds_path + 'val/',   shuffle=False)

BATCH = params['training']['batch_size']
train_loader = DataLoader(train_ds, batch_size=BATCH, num_workers=4, collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, num_workers=2, collate_fn=collate_fn)

print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

In [ ]:
# ── Cell 6: Build model (Stage 1 — backbone frozen) ──────────────────────
from transformers import AutoModelForObjectDetection

device = torch.device('cuda')

model = AutoModelForObjectDetection.from_pretrained(
    'microsoft/conditional-detr-resnet-50',
    num_labels=num_classes,
    ignore_mismatched_sizes=True,
).to(device)

# Freeze backbone — Stage 1 coarse training
for param in model.model.backbone.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)')
print('Backbone frozen. Stage 1 coarse ready.')

In [ ]:
# ── Cell 7: Training loop ─────────────────────────────────────────────────
import time, boto3

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr, weight_decay=params['training']['weight_decay']
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=params['training']['lr_drop_epoch'], gamma=0.1
)

@torch.no_grad()
def evaluate(loader, max_steps=20):
    model.eval()
    total_loss, total_conf, steps = 0, 0, 0
    for images, anns in loader:
        out = model(pixel_values=images.to(device), labels=anns)
        total_loss += out.loss.item()
        scores = torch.softmax(out.logits, -1)[:, :, :-1].max(-1).values
        total_conf += scores.mean().item()
        steps += 1
        if steps >= max_steps: break
    return total_loss / steps, total_conf / steps

best_val_loss = float('inf')

with mlflow.start_run(run_name=f'detr-stage1-coarse-{dataset_version}') as run:
    mlflow.log_params({
        'model': 'conditional-detr-resnet-50', 'stage': 'coarse',
        'image_size': image_size, 'backbone_frozen': True,
        'epochs': epochs, 'lr': lr, 'batch_size': BATCH,
        'mds_path': mds_path, 'dataset_version': dataset_version,
        'num_classes': num_classes,
    })

    for epoch in range(epochs):
        model.train()
        epoch_loss, t0 = 0, time.time()

        for images, anns in train_loader:
            out = model(pixel_values=images.to(device), labels=anns)
            optimizer.zero_grad()
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), params['training']['gradient_clip'])
            optimizer.step()
            epoch_loss += out.loss.item()

        scheduler.step()
        train_loss = epoch_loss / len(train_loader)
        val_loss, val_conf = evaluate(val_loader)

        mlflow.log_metrics({
            'train_loss': train_loss, 'val_loss': val_loss,
            'val_mean_confidence': val_conf, 'lr': scheduler.get_last_lr()[0]
        }, step=epoch)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), '/tmp/detr_best.pt')

        print(f'Epoch {epoch+1:3d}/{epochs} | train={train_loss:.4f} | val={val_loss:.4f} | conf={val_conf:.3f} | {time.time()-t0:.0f}s')

    RUN_ID = run.info.run_id
    print(f'\nBest val_loss: {best_val_loss:.4f}')
    print(f'Run ID: {RUN_ID}')

In [ ]:
# ── Cell 8: Upload weights to S3 + log to MLflow ─────────────────────────
S3_BUCKET = 'my-perception-robops-data-2026-688567275774-eu-central-1-an'
s3_key    = f'weights/detr/{dataset_version}/model.pt'

s3 = boto3.client('s3', region_name='eu-central-1')
s3.upload_file('/tmp/detr_best.pt', S3_BUCKET, s3_key)
s3_url = f's3://{S3_BUCKET}/{s3_key}'
print(f'Weights uploaded → {s3_url}')

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param('s3_weights_url', s3_url)
    mlflow.pytorch.log_model(model, 'model')
    print('Model logged to MLflow')

In [ ]:
# ── Cell 9: Register model in MLflow as Staging ───────────────────────────
client     = mlflow.tracking.MlflowClient()
model_name = 'detr-conditional-resnet50'
model_uri  = f'runs:/{RUN_ID}/model'

try:
    client.create_registered_model(model_name)
except Exception:
    pass  # already exists

version = client.create_model_version(
    name=model_name, source=model_uri, run_id=RUN_ID,
    tags={'stage': 'coarse', 'architecture': 'detr', 'dataset_version': dataset_version}
)
client.transition_model_version_stage(name=model_name, version=version.version, stage='Staging')

print(f'\n✅ Model registered as Staging (version={version.version})')
print(f'\n📋 Copy this Run ID for GitHub Actions ci_deploy.yml:')
print(f'   RUN_ID = {RUN_ID}')
print(f'   Dataset version = {dataset_version}')
print(f'\n🔗 View experiment: https://dagshub.com/srnortw/learning-robotic-perception-robops/experiments')

In [ ]:
# ── Cell 10: Save drift baseline (for Phase B Round 2+) ──────────────────
import json
from collections import Counter
from pathlib import Path

class_counts = Counter()
total_conf, steps = 0, 0

model.eval()
with torch.no_grad():
    for images, anns in val_loader:
        out = model(pixel_values=images.to(device), labels=anns)
        scores = torch.softmax(out.logits, -1)[:, :, :-1].max(-1).values
        total_conf += scores.mean().item()
        for ann_list in anns:
            for ann in ann_list:
                class_counts[str(ann.get('category_id', 0))] += 1
        steps += 1
        if steps >= 50: break

total = sum(class_counts.values()) or 1
baseline = {
    'architecture': 'detr',
    'dataset_version': dataset_version,
    'class_frequency': {k: v / total for k, v in class_counts.items()},
    'mean_confidence': round(total_conf / max(steps, 1), 4),
    'mean_brightness': 128.0,
}

baseline_path = Path('pipeline/phase_b/drift_baseline_detr.json')
with open(baseline_path, 'w') as f:
    json.dump(baseline, f, indent=2)

print(f'Drift baseline saved → {baseline_path}')
print(json.dumps(baseline, indent=2))
print('\nNext: git add pipeline/phase_b/drift_baseline_detr.json && git commit && git push')

## After Training — Trigger GitHub Actions

1. Copy the `RUN_ID` printed in Cell 9
2. Go to → [GitHub Actions](https://github.com/srnortw/learning-robotic-perception-robops/actions)
3. Select **CI — ONNX Export, Quantize, Push to ECR**
4. Click **Run workflow** → paste `RUN_ID` + dataset version
5. This auto-runs: export → INT8 quantize → ECR push → Discord notify Phase D

---
**MLflow dashboard:** https://dagshub.com/srnortw/learning-robotic-perception-robops/experiments